In [49]:
%pip install pandas requests fredapi matplotlib python-dotenv eurostat


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# ENTSOG Weekly Country Panel

Builds weekly country panels for DE, FR, IT, NL, ES, UK and an EU aggregate from ENTSOG AggregatedData.

Pipeline behavior:
- First run: backfill from 2018-01-01
- Later runs: incremental fetch from the last stored date with overlap buffer
- Primary indicators: Physical Flow and Allocation
- Units: convert daily kWh/d values into weekly TWh
- Derived metrics: level, WoW %, YoY %

## Setup

**FRED API Key required:** Create a `.env` file in the workspace root by copying `.env.example`:
```bash
cp .env.example .env
```
Then edit `.env` and add your FRED API key from https://fred.stlouisfed.org/docs/api/

In [50]:
from __future__ import annotations

import os
import time
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

# -----------------------------
# Config
# -----------------------------
BASE_URL = "https://transparency.entsog.eu/api/v1/AggregatedData"
COUNTRIES = ["DE", "FR", "IT", "NL", "ES", "UK"]
INDICATORS = ["Physical Flow", "Allocation"]

START_DATE = date(2018, 1, 1)
FORCE_MAX_HISTORY_PULL = True
OVERLAP_DAYS = 7
TIMEZONE = "CET"
PERIOD_TYPE = "day"
PAGE_LIMIT = 1000
MAX_RETRIES = 4
RETRY_BACKOFF_SECONDS = 1.5
REQUEST_TIMEOUT_SECONDS = 60

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "methodo" else Path.cwd().resolve()
OUTPUT_DIR = ROOT / "data" / "processed"
DAILY_OUTPUT = OUTPUT_DIR / "entsog_daily_selected.csv"
WEEKLY_OUTPUT = OUTPUT_DIR / "entsog_weekly_panel.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")
print(f"Daily output: {DAILY_OUTPUT}")
print(f"Weekly output: {WEEKLY_OUTPUT}")
print(f"Force max-history pull: {FORCE_MAX_HISTORY_PULL}")

Root: /workspaces/high_frequency
Daily output: /workspaces/high_frequency/data/processed/entsog_daily_selected.csv
Weekly output: /workspaces/high_frequency/data/processed/entsog_weekly_panel.csv
Force max-history pull: True


In [51]:
from dotenv import load_dotenv
import matplotlib.pyplot as plt

# Load environment variables from .env file
ENV_FILE = ROOT.parent / ".env"
if ENV_FILE.exists():
    load_dotenv(ENV_FILE)
    print(f"✓ Loaded .env file from {ENV_FILE}")
else:
    print(f"⚠ .env file not found at {ENV_FILE}")
    print(f"  Create a .env file by copying .env.example and filling in your FRED_API_KEY")

⚠ .env file not found at /workspaces/.env
  Create a .env file by copying .env.example and filling in your FRED_API_KEY


In [52]:
def _safe_get_json(session: requests.Session, url: str, params: dict) -> dict:
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)

            # ENTSOG returns 404 JSON messages for "No result found" and archive limits.
            if resp.status_code == 404:
                return {}

            resp.raise_for_status()
            return resp.json()
        except Exception as exc:
            last_err = exc
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS * attempt)
            else:
                raise RuntimeError(f"Request failed after {MAX_RETRIES} attempts. params={params}") from last_err


def _month_start(d: date) -> date:
    return date(d.year, d.month, 1)


def _next_month(d: date) -> date:
    if d.month == 12:
        return date(d.year + 1, 1, 1)
    return date(d.year, d.month + 1, 1)


def _resolve_available_start_date(
    session: requests.Session,
    requested_start: date,
    end_date: date,
    probe_indicator: str,
) -> date:
    # Probe monthly windows and return the first month that yields data.
    probe = _month_start(requested_start)
    while probe <= end_date:
        nxt = _next_month(probe)
        window_end = min(nxt - timedelta(days=1), end_date)
        params = {
            "from": probe.isoformat(),
            "to": window_end.isoformat(),
            "indicator": probe_indicator,
            "periodType": PERIOD_TYPE,
            "timeZone": TIMEZONE,
            "limit": PAGE_LIMIT,
            "offset": 0,
        }
        payload = _safe_get_json(session, BASE_URL, params)
        chunk = payload.get("AggregatedData", payload.get("aggregatedData", []))
        if chunk:
            return probe
        probe = nxt

    return requested_start


def fetch_aggregated_data(
    start_date: date,
    end_date: date,
    countries: list[str],
    indicators: list[str],
) -> pd.DataFrame:
    rows: list[dict] = []
    session = requests.Session()

    effective_start = _resolve_available_start_date(
        session=session,
        requested_start=start_date,
        end_date=end_date,
        probe_indicator=indicators[0],
    )

    print(f"Effective historical start used by API: {effective_start}")

    # Pull month-by-month to avoid sparse/partial results from huge windows.
    for indicator in indicators:
        cursor = _month_start(effective_start)
        while cursor <= end_date:
            nxt = _next_month(cursor)
            window_end = min(nxt - timedelta(days=1), end_date)

            offset = 0
            while True:
                params = {
                    "from": cursor.isoformat(),
                    "to": window_end.isoformat(),
                    "indicator": indicator,
                    "periodType": PERIOD_TYPE,
                    "timeZone": TIMEZONE,
                    "limit": PAGE_LIMIT,
                    "offset": offset,
                }

                payload = _safe_get_json(session, BASE_URL, params)
                chunk = payload.get("AggregatedData", payload.get("aggregatedData", []))
                if not chunk:
                    break

                rows.extend(chunk)
                if len(chunk) < PAGE_LIMIT:
                    break

                offset += PAGE_LIMIT

            cursor = nxt

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows)

In [53]:
# -----------------------------
# Incremental date window
# -----------------------------
today = date.today()

if DAILY_OUTPUT.exists():
    existing_daily = pd.read_csv(DAILY_OUTPUT)
else:
    existing_daily = pd.DataFrame()

if FORCE_MAX_HISTORY_PULL:
    last_date = pd.to_datetime(existing_daily["date"], errors="coerce").dt.date.max() if (not existing_daily.empty and "date" in existing_daily.columns) else None
    fetch_start = START_DATE
else:
    if not existing_daily.empty and "date" in existing_daily.columns:
        existing_daily["date"] = pd.to_datetime(existing_daily["date"], errors="coerce").dt.date
        last_date = existing_daily["date"].max()
        fetch_start = max(START_DATE, last_date - timedelta(days=OVERLAP_DAYS))
    else:
        last_date = None
        fetch_start = START_DATE

print(f"Last stored date: {last_date}")
print(f"Fetch window: {fetch_start} -> {today}")

new_raw = fetch_aggregated_data(fetch_start, today, COUNTRIES, INDICATORS)
print(f"Fetched rows: {len(new_raw):,}")

if new_raw.empty and existing_daily.empty:
    raise RuntimeError("No data fetched and no existing local data available.")

if not new_raw.empty:
    keep_cols = [
        "id",
        "countryKey",
        "countryLabel",
        "indicator",
        "periodType",
        "periodFrom",
        "periodTo",
        "unit",
        "value",
        "flowStatus",
        "lastUpdateDateTime",
        "directionKey",
        "adjacentSystemsLabel",
        "operatorLabel",
        "bzShort",
    ]
    existing_cols = [c for c in keep_cols if c in new_raw.columns]
    new_daily = new_raw[existing_cols].copy()

    new_daily["date"] = pd.to_datetime(new_daily["periodFrom"], errors="coerce", utc=True).dt.date
    new_daily["value"] = pd.to_numeric(new_daily["value"], errors="coerce")
    new_daily["lastUpdateDateTime"] = pd.to_datetime(new_daily["lastUpdateDateTime"], errors="coerce", utc=True)

    new_daily = new_daily[new_daily["countryKey"].isin(COUNTRIES)]
    new_daily = new_daily[new_daily["indicator"].isin(INDICATORS)]
    new_daily = new_daily[new_daily["value"].notna()]
else:
    new_daily = pd.DataFrame()

if existing_daily.empty:
    combined_daily = new_daily.copy()
else:
    combined_daily = pd.concat([existing_daily, new_daily], ignore_index=True, sort=False)

if not combined_daily.empty:
    combined_daily["lastUpdateDateTime"] = pd.to_datetime(combined_daily["lastUpdateDateTime"], errors="coerce", utc=True)
    combined_daily = combined_daily.sort_values(["id", "lastUpdateDateTime"])
    # Keep the latest revision per ENTSOG id
    if "id" in combined_daily.columns:
        combined_daily = combined_daily.drop_duplicates(subset=["id"], keep="last")

# Compute EU daily aggregate (sum across 6 countries per indicator per day)
eu_daily = (
    combined_daily[combined_daily["countryKey"].isin(COUNTRIES)]
    .groupby(["date", "indicator", "periodType", "unit"], as_index=False)["value"]
    .sum()
    .assign(
        countryKey="EU",
        countryLabel="EU Aggregate",
        id=None,
        flowStatus=None,
        lastUpdateDateTime=None,
        directionKey=None,
        adjacentSystemsLabel=None,
        operatorLabel=None,
        bzShort=None,
        periodFrom=None,
        periodTo=None,
    )
)

# Reorder columns to match combined_daily
eu_daily = eu_daily[combined_daily.columns]

# Append EU aggregate to combined daily
combined_daily = pd.concat([combined_daily, eu_daily], ignore_index=True, sort=False)
combined_daily = combined_daily.sort_values(["date", "countryKey", "indicator"]).reset_index(drop=True)

combined_daily.to_csv(DAILY_OUTPUT, index=False)
print(f"Saved daily dataset: {len(combined_daily):,} rows (including EU aggregate)")
print(f"Countries in daily output: {sorted(combined_daily['countryKey'].dropna().unique().tolist())}")

Last stored date: 2026-04-29
Fetch window: 2018-01-01 -> 2026-04-29
Effective historical start used by API: 2019-11-01
Fetched rows: 104,281
Saved daily dataset: 48,048 rows (including EU aggregate)
Countries in daily output: ['DE', 'ES', 'EU', 'FR', 'IT', 'NL', 'UK']


In [54]:
# -----------------------------
# Weekly panel construction
# -----------------------------
DROP_INCOMPLETE_LAST_WEEK = True


daily = combined_daily.copy()
daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
daily = daily.dropna(subset=["date", "value", "countryKey", "indicator"])

# Monday-start week key
daily["week_start"] = daily["date"] - pd.to_timedelta(daily["date"].dt.weekday, unit="D")
daily["twh"] = daily["value"] / 1_000_000_000.0

country_weekly = (
    daily.groupby(["countryKey", "indicator", "week_start"], as_index=False)["twh"]
    .sum()
    .rename(columns={"twh": "twh_week"})
)

# Reindex to contiguous weekly frequency per country/indicator,
# so WoW/YoY do not compare across multi-year gaps.
parts = []
for (country, indicator), g in country_weekly.groupby(["countryKey", "indicator"], as_index=False):
    g = g.sort_values("week_start")
    full_weeks = pd.date_range(g["week_start"].min(), g["week_start"].max(), freq="W-MON")
    gg = g.set_index("week_start").reindex(full_weeks)
    gg.index.name = "week_start"
    gg = gg.reset_index()
    gg["countryKey"] = country
    gg["indicator"] = indicator
    parts.append(gg[["countryKey", "indicator", "week_start", "twh_week"]])

country_weekly = pd.concat(parts, ignore_index=True)

if DROP_INCOMPLETE_LAST_WEEK:
    current_week_start = pd.Timestamp.today().normalize() - pd.to_timedelta(pd.Timestamp.today().weekday(), unit="D")
    country_weekly = country_weekly[country_weekly["week_start"] < current_week_start]

eu_weekly = (
    country_weekly.groupby(["indicator", "week_start"], as_index=False)["twh_week"]
    .sum(min_count=1)
    .assign(countryKey="EU")
)
eu_weekly = eu_weekly[["countryKey", "indicator", "week_start", "twh_week"]]

panel = pd.concat([country_weekly, eu_weekly], ignore_index=True)
panel = panel.sort_values(["countryKey", "indicator", "week_start"]).reset_index(drop=True)

panel["wow_pct"] = panel.groupby(["countryKey", "indicator"])["twh_week"].pct_change(1)
panel["yoy_pct"] = panel.groupby(["countryKey", "indicator"])["twh_week"].pct_change(52)

panel.to_csv(WEEKLY_OUTPUT, index=False)

print(f"Saved weekly panel: {len(panel):,} rows")
print("Countries in output:", sorted(panel["countryKey"].dropna().unique().tolist()))
print("Indicators in output:", sorted(panel["indicator"].dropna().unique().tolist()))

panel.tail(20)

Saved weekly panel: 5,388 rows
Countries in output: ['DE', 'ES', 'EU', 'FR', 'IT', 'NL', 'UK']
Indicators in output: ['Allocation', 'Physical Flow']


,countryKey,indicator,week_start,twh_week,wow_pct,yoy_pct
5368,UK,Physical Flow,2025-12-08,NaN,NaN,NaN
5369,UK,Physical Flow,2025-12-15,NaN,NaN,NaN
5370,UK,Physical Flow,2025-12-22,NaN,NaN,NaN
5371,UK,Physical Flow,2025-12-29,11.754276,NaN,1.123306
5372,UK,Physical Flow,2026-01-05,NaN,NaN,NaN
5373,UK,Physical Flow,2026-01-12,NaN,NaN,NaN
5374,UK,Physical Flow,2026-01-19,NaN,NaN,NaN
5375,UK,Physical Flow,2026-01-26,5.535827,NaN,-0.080135
5376,UK,Physical Flow,2026-02-02,5.641220,0.019038,NaN
5377,UK,Physical Flow,2026-02-09,NaN,NaN,NaN


# FRED Locked Universe Fetch Only

This section only fetches the previously agreed FRED series universe (locked metadata list) and saves observations.

No extra processing, plotting, or derived indices are performed in this section.

In [63]:
import os
from dotenv import load_dotenv

# Load API key
ENV_FILE = ROOT / ".env"
if ENV_FILE.exists():
    load_dotenv(ENV_FILE)
    print(f"Loaded .env from {ENV_FILE}")
else:
    print(f".env file not found at {ENV_FILE}")

FRED_API_KEY = os.getenv("FRED_API_KEY")
if not FRED_API_KEY:
    raise RuntimeError("FRED_API_KEY environment variable not set.")

# Locked universe files
UNIVERSE_METADATA_OUTPUT = OUTPUT_DIR / "fred_europe_macro_energy_universe_metadata.csv"
UNIVERSE_OBS_OUTPUT = OUTPUT_DIR / "fred_europe_macro_energy_universe_observations.csv"
UNIVERSE_ERRORS_OUTPUT = OUTPUT_DIR / "fred_europe_macro_energy_universe_errors.csv"

if not UNIVERSE_METADATA_OUTPUT.exists():
    raise RuntimeError(
        f"Locked universe file not found: {UNIVERSE_METADATA_OUTPUT}. "
        "Run the universe discovery/filter step once before this fetch-only workflow."
    )

meta_locked = pd.read_csv(UNIVERSE_METADATA_OUTPUT)
locked_series_ids = (
    meta_locked["id"].dropna().astype(str).drop_duplicates().sort_values().tolist()
)

FRED_API_ROOT = "https://api.stlouisfed.org/fred"
OBS_START_DATE = date(1980, 1, 1)

print(f"Locked series to fetch: {len(locked_series_ids):,}")
print(f"Observations output: {UNIVERSE_OBS_OUTPUT}")

Loaded .env from /workspaces/high_frequency/.env
Locked series to fetch: 515
Observations output: /workspaces/high_frequency/data/processed/fred_europe_macro_energy_universe_observations.csv


In [64]:
from urllib.parse import urlencode


def _fred_get(endpoint: str, params: dict) -> dict:
    q = dict(params)
    q["api_key"] = FRED_API_KEY
    q["file_type"] = "json"
    url = f"{FRED_API_ROOT}/{endpoint}?{urlencode(q)}"

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, timeout=REQUEST_TIMEOUT_SECONDS)
            if resp.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"Retryable HTTP {resp.status_code}")
            resp.raise_for_status()
            return resp.json()
        except Exception as exc:
            last_err = exc
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_BACKOFF_SECONDS * attempt)
            else:
                raise RuntimeError(f"FRED request failed after {MAX_RETRIES} attempts: {endpoint}") from last_err


obs_parts = []
errors = []

for i, sid in enumerate(locked_series_ids, start=1):
    try:
        payload = _fred_get(
            "series/observations",
            {
                "series_id": sid,
                "observation_start": OBS_START_DATE.isoformat(),
                "sort_order": "asc",
            },
        )
        obs = pd.DataFrame(payload.get("observations", []))
        if obs.empty:
            continue

        obs = obs[[c for c in ["date", "value", "realtime_start", "realtime_end"] if c in obs.columns]].copy()
        obs["series_id"] = sid
        obs["date"] = pd.to_datetime(obs["date"], errors="coerce")
        obs["value"] = pd.to_numeric(obs["value"], errors="coerce")
        obs_parts.append(obs)

        if i % 100 == 0:
            print(f"Downloaded {i:,}/{len(locked_series_ids):,} series...")
    except Exception as exc:
        errors.append({"series_id": sid, "error": str(exc)})

if not obs_parts:
    raise RuntimeError("No observations downloaded for the locked FRED universe.")

obs_all = pd.concat(obs_parts, ignore_index=True)
obs_all = obs_all.sort_values(["series_id", "date"]).reset_index(drop=True)
obs_all.to_csv(UNIVERSE_OBS_OUTPUT, index=False)

if errors:
    pd.DataFrame(errors).to_csv(UNIVERSE_ERRORS_OUTPUT, index=False)
    print(f"Completed with {len(errors):,} series errors. See: {UNIVERSE_ERRORS_OUTPUT}")
else:
    print("Completed with no series errors.")

print(f"Saved observations: {UNIVERSE_OBS_OUTPUT}")
print(f"Series downloaded: {obs_all['series_id'].nunique():,}")
print(f"Observation rows: {len(obs_all):,}")
print(f"Date range: {obs_all['date'].min()} to {obs_all['date'].max()}")

obs_all.head()

Downloaded 100/515 series...
Downloaded 200/515 series...
Downloaded 300/515 series...
Downloaded 400/515 series...
Downloaded 500/515 series...
Completed with 9 series errors. See: /workspaces/high_frequency/data/processed/fred_europe_macro_energy_universe_errors.csv
Saved observations: /workspaces/high_frequency/data/processed/fred_europe_macro_energy_universe_observations.csv
Series downloaded: 506
Observation rows: 372,867
Date range: 1987-05-01 00:00:00 to 2026-04-28 00:00:00


,date,value,realtime_start,realtime_end,series_id
0,1996-01-01,NaN,2026-04-29,2026-04-29,00XE00ATM086NEST
1,1996-02-01,NaN,2026-04-29,2026-04-29,00XE00ATM086NEST
2,1996-03-01,NaN,2026-04-29,2026-04-29,00XE00ATM086NEST
3,1996-04-01,NaN,2026-04-29,2026-04-29,00XE00ATM086NEST
4,1996-05-01,NaN,2026-04-29,2026-04-29,00XE00ATM086NEST


# Eurostat Locked Additions Fetch Only (Step 3)

This section fetches only the 14 pre-approved Eurostat datasets identified as likely new versus the locked FRED pull.

It saves a normalized long dataset plus an error log in data/processed.

In [4]:
import re
from pathlib import Path

import eurostat
import pandas as pd

# Allow this section to run independently from earlier setup cells.
if "OUTPUT_DIR" not in globals():
    ROOT = Path.cwd().resolve().parent if Path.cwd().name == "methodo" else Path.cwd().resolve()
    OUTPUT_DIR = ROOT / "data" / "processed"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Initialized OUTPUT_DIR: {OUTPUT_DIR}")

EUROSTAT_LOCKED_CODES = [
    "STS_TRTU_M",
    "EI_ISRR_M",
    "EI_ISRT_M",
    "EI_BSRT_M_R2",
    "STS_TRLB_M",
    "STS_INPPD_M",
    "STS_INPPND_M",
    "STS_INPI_M",
    "STS_INPP_M",
    "PRC_FSC_IDX",
    "PRC_IPC_G20",
    "STS_COPI_M",
    "NRG_CB_GASM",
    "EI_CPHI_M",
]

EUROSTAT_CODES_OUTPUT = OUTPUT_DIR / "eurostat_likely_new_locked_codes.csv"
EUROSTAT_OBS_OUTPUT = OUTPUT_DIR / "eurostat_likely_new_observations.csv"
EUROSTAT_ERRORS_OUTPUT = OUTPUT_DIR / "eurostat_likely_new_errors.csv"

pd.DataFrame({"code": EUROSTAT_LOCKED_CODES}).to_csv(EUROSTAT_CODES_OUTPUT, index=False)
print(f"Locked Eurostat datasets: {len(EUROSTAT_LOCKED_CODES):,}")
print(f"Codes output: {EUROSTAT_CODES_OUTPUT}")

Locked Eurostat datasets: 14
Codes output: /workspaces/high_frequency/data/processed/eurostat_likely_new_locked_codes.csv


In [5]:
time_col_pattern = r"^\d{4}(?:-\d{2})?(?:-\d{2})?$"
preferred_geos = ["EU27_2020", "EA20", "EA19", "EU28", "EU27_2007"]
narrow_dims_order = ["unit", "indic", "na_item", "s_adj", "nace_r2", "nrg_bal", "product", "cpa2_1"]

def _normalize_eurostat_df(df: pd.DataFrame, code: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    d = df.copy()
    time_cols = [c for c in d.columns if re.fullmatch(time_col_pattern, str(c))]
    id_cols = [c for c in d.columns if c not in time_cols]

    if not time_cols:
        return pd.DataFrame()

    long_df = d.melt(id_vars=id_cols, value_vars=time_cols, var_name="time", value_name="value")
    long_df["dataset_code"] = code
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df[long_df["value"].notna()].copy()
    long_df["date"] = pd.to_datetime(long_df["time"], errors="coerce")
    return long_df

def _safe_get_values(code: str, dim: str) -> list[str]:
    try:
        vals = eurostat.get_par_values(code, dim)
        return [v for v in vals if isinstance(v, str)]
    except Exception:
        return []

def _build_filter_variants(code: str) -> list[dict]:
    try:
        dims_df = eurostat.get_dic(code, frmt="df")
        dims = [str(x).lower() for x in dims_df["dim"].tolist()]
    except Exception:
        dims = []

    # Start with strongly size-constrained windows to avoid 413 extraction limits.
    base_windows = [
        {"startPeriod": 2025, "endPeriod": 2026},
        {"startPeriod": 2020, "endPeriod": 2026},
        {"startPeriod": 2018, "endPeriod": 2026},
    ]

    variants = []
    for base in base_windows:
        b = dict(base)
        if "freq" in dims:
            b["freq"] = "M"

        if "geo" in dims:
            geo_vals = _safe_get_values(code, "geo")
            chosen_geo = next((g for g in preferred_geos if g in geo_vals), None)
            if chosen_geo is None and geo_vals:
                chosen_geo = geo_vals[0]
            if chosen_geo:
                b["geo"] = chosen_geo

        variants.append(b.copy())

        current = b.copy()
        for dim in narrow_dims_order:
            if dim in dims and dim not in current:
                vals = _safe_get_values(code, dim)
                if vals:
                    current = {**current, dim: vals[0]}
                    variants.append(current.copy())

    # Last resort: minimal recent window only.
    variants.append({"startPeriod": 2025, "endPeriod": 2026})

    uniq = []
    seen = set()
    for v in variants:
        key = tuple(sorted(v.items()))
        if key not in seen:
            seen.add(key)
            uniq.append(v)
    return uniq

parts = []
errors = []

for i, code in enumerate(EUROSTAT_LOCKED_CODES, start=1):
    done = False
    last_exc = None
    variants = _build_filter_variants(code)

    for filt in variants:
        try:
            df_code = eurostat.get_data_df(code, flags=False, filter_pars=filt)
            norm = _normalize_eurostat_df(df_code, code)
            if not norm.empty:
                norm["applied_filter"] = str(filt)
                parts.append(norm)
                done = True
                break
        except Exception as exc:
            last_exc = exc
            continue

    if not done:
        err_msg = str(last_exc) if last_exc else "No parsable time/value data"
        errors.append({"code": code, "error": err_msg})

    if i % 5 == 0 or i == len(EUROSTAT_LOCKED_CODES):
        print(f"Fetched {i:,}/{len(EUROSTAT_LOCKED_CODES):,} Eurostat datasets...")

if parts:
    eurostat_obs = pd.concat(parts, ignore_index=True)
    eurostat_obs = eurostat_obs.sort_values(["dataset_code", "date", "time"]).reset_index(drop=True)
else:
    eurostat_obs = pd.DataFrame(columns=["dataset_code", "time", "date", "value", "applied_filter"])

eurostat_obs.to_csv(EUROSTAT_OBS_OUTPUT, index=False)

if errors:
    pd.DataFrame(errors).to_csv(EUROSTAT_ERRORS_OUTPUT, index=False)
    print(f"Completed with {len(errors):,} dataset errors. See: {EUROSTAT_ERRORS_OUTPUT}")
else:
    print("Completed with no dataset errors.")

print(f"Saved Eurostat observations: {EUROSTAT_OBS_OUTPUT}")
print(f"Datasets with parsed observations: {eurostat_obs['dataset_code'].nunique():,}")
print(f"Observation rows: {len(eurostat_obs):,}")
if not eurostat_obs.empty:
    print(f"Date range: {eurostat_obs['date'].min()} to {eurostat_obs['date'].max()}")

eurostat_obs.head()

Fetched 5/14 Eurostat datasets...
Fetched 10/14 Eurostat datasets...
Fetched 14/14 Eurostat datasets...
Completed with no dataset errors.
Saved Eurostat observations: /workspaces/high_frequency/data/processed/eurostat_likely_new_observations.csv
Datasets with parsed observations: 14
Observation rows: 50,591
Date range: 2025-01-01 00:00:00 to 2026-04-01 00:00:00


,freq,indic_bt,nace_r2,s_adj,unit,geo\TIME_PERIOD,time,value,dataset_code,date,applied_filter,indic,cpa2_1,indx,coicop,nrg_bal,siec
0,M,NaN,NaN,NSA,BAL,EU27_2020,2025-01,13.0,EI_BSRT_M_R2,2025-01-01,"{'startPeriod': 2025, 'endPeriod': 2026, 'freq...",BS-RAS,NaN,NaN,NaN,NaN,NaN
1,M,NaN,NaN,SA,BAL,EU27_2020,2025-01,12.6,EI_BSRT_M_R2,2025-01-01,"{'startPeriod': 2025, 'endPeriod': 2026, 'freq...",BS-RAS,NaN,NaN,NaN,NaN,NaN
2,M,NaN,NaN,NSA,BAL,EU27_2020,2025-01,-6.4,EI_BSRT_M_R2,2025-01-01,"{'startPeriod': 2025, 'endPeriod': 2026, 'freq...",BS-RCI,NaN,NaN,NaN,NaN,NaN
3,M,NaN,NaN,SA,BAL,EU27_2020,2025-01,-4.0,EI_BSRT_M_R2,2025-01-01,"{'startPeriod': 2025, 'endPeriod': 2026, 'freq...",BS-RCI,NaN,NaN,NaN,NaN,NaN
4,M,NaN,NaN,NSA,BAL,EU27_2020,2025-01,-12.6,EI_BSRT_M_R2,2025-01-01,"{'startPeriod': 2025, 'endPeriod': 2026, 'freq...",BS-REBS,NaN,NaN,NaN,NaN,NaN
